# Filename Renaming Template

> **Note:** Notebooks in this directory use CLI commands rather than Python library imports.

This notebook creates a mapping CSV of original S3 filenames to new filenames without processing the files.

## Output CSV Columns
- `original_s3_path` - Full S3 path to original file
- `original_filename` - Original filename only
- `new_filename` - Proposed new filename
- `category` - File category
- `file_size_gb` - File size in GB
- `nodata_value` - Nodata value for the category
- `status` - Validation status
- `output_s3_path` - Proposed destination path

---

## Step 1: Configuration

Set your event details and S3 paths:

In [ ]:
import re
import os
import subprocess
import pandas as pd
from pathlib import Path

# ========================================
# INPUTS
# ========================================

# S3 Paths (DO NOT CHANGE)
BUCKET = 'nasa-disasters'    # S3 bucket
DESTINATION_BASE = 'drcs_activations_new'  # Where to save COGs in S3 bucket
GEOTIFF_DIR = 'drcs_activations' # This is where all raw geotiff files currently are

# Event Details
EVENT_NAME = '202510_Flood_AK'  # Your event name
SOURCE = "TBD"  # Data origin (e.g., USGS, Copernicus, CSDA, TBD)
SUB_PRODUCT_NAME = 'sentinel2'         # Sub-directories within EVENT_NAME (RGB, trueColor, SWIR, etc.). Can leave blank.
SOURCE_PATH = f'{GEOTIFF_DIR}/{EVENT_NAME}/{SUB_PRODUCT_NAME}'      # Where your files are

# Output Options
SAVE_CSV = True          # Save mapping to CSV file
OUTPUT_DIR = '/tmp/file-mapping'    # Local directory for CSV output

print(f"Event: {EVENT_NAME}")
print(f"Source: s3://{BUCKET}/{SOURCE_PATH}")

assert re.match(r'^\d{6}_\w+_\w+$', EVENT_NAME), \
    f"EVENT_NAME must follow format: YYYYMM_Text_Text, got: '{EVENT_NAME}'"

In [ ]:
from shared_utils import PROCESSOR_STRING
ACTIVATION_METADATA = {
    "ACTIVATION_EVENT": EVENT_NAME,
    "SOURCE": SOURCE,
    "PROCESSOR": PROCESSOR_STRING,
    # Add any custom key-value pairs here
}


## Step 2: List S3 Files

Use `aws s3 ls` to list available files with their sizes:

In [ ]:
# List all files in the S3 path using aws cli
s3_uri = f"s3://{BUCKET}/{SOURCE_PATH}/"
print(f"Listing files in {s3_uri}")
print("=" * 80)

result = subprocess.run(
    ['aws', 's3', 'ls', s3_uri, '--recursive'],
    capture_output=True, text=True
)

if result.returncode != 0:
    print(f"Error listing S3 files: {result.stderr}")
    files_df = pd.DataFrame()
else:
    # Parse aws s3 ls output
    # Format: "2025-09-15 12:34:56  123456789 path/to/file.tif"
    file_data = []
    for line in result.stdout.strip().splitlines():
        parts = line.split()
        if len(parts) >= 4:
            size_bytes = int(parts[2])
            file_path = parts[3]
            filename = os.path.basename(file_path)

            # Only include .tif files
            if filename.lower().endswith('.tif'):
                file_data.append({
                    'original_s3_path': file_path,
                    'original_filename': filename,
                    'file_size_gb': round(size_bytes / (1024 ** 3), 6)
                })

    if file_data:
        files_df = pd.DataFrame(file_data)
        print(f"Found {len(files_df)} .tif files\n")
        print(f"Total size: {files_df['file_size_gb'].sum():.2f} GB\n")

        print("Complete file list:")
        print("-" * 80)
        for i, row in files_df.iterrows():
            print(f"{i+1:3}. {row['original_filename']:<65} ({row['file_size_gb']:.4f} GB)")
    else:
        print("No .tif files found in the specified path.")
        print("   Check your SOURCE_PATH configuration.")
        files_df = pd.DataFrame()

## Step 3: Define Filename Transformation Functions

Based on the files you see above, configure:
1. **Categorization patterns** - Regex patterns to identify file types
2. **Filename functions** - How to transform filenames
3. **Output directories** - Where each category should be saved

In [ ]:
# ========================================
# HELPER FUNCTIONS
# ========================================
#
# Canonical filename helpers live in shared_utils.file_naming. The wrappers
# below preserve this notebook's legacy return signatures (YYYY-MM-DD dates,
# TC/SWIR/NC/CIR naming) while delegating the real work to the shared module.

import re
import os
import sys
from pathlib import Path

# Ensure the repo root is on sys.path so shared_utils is importable when the
# notebook is run from the notebooks/ directory.
_repo_root = Path('.').resolve()
for _ in range(5):
    if (_repo_root / 'shared_utils').is_dir():
        break
    _repo_root = _repo_root.parent
if str(_repo_root) not in sys.path:
    sys.path.insert(0, str(_repo_root))

from shared_utils.file_naming import (
    extract_datetime_from_filename,
    create_output_filename as _create_output_filename,
)


def extract_date_from_filename(filename):
    """Extract date from filename in YYYY-MM-DD format.

    Thin wrapper around shared_utils.file_naming.extract_datetime_from_filename
    that preserves this notebook's legacy YYYY-MM-DD return contract.
    """
    matched, _granularity = extract_datetime_from_filename(filename)
    if not matched:
        return None
    if len(matched) == 8 and matched.isdigit():
        return f"{matched[0:4]}-{matched[4:6]}-{matched[6:8]}"
    return matched


def create_TC_SWIR_NC_CIR_filename(original_path, event_name):
    """Create filename for trueColor/SWIR/naturalColor/colorInfrared products.

    Delegates to shared_utils.file_naming.create_output_filename.
    """
    return _create_output_filename(original_path, event_name)


In [ ]:
# Test the function with a sample filename
oldName = 'S2B_MSIL2A_colorInfrared_20250913_222529_T03VVG'
print(f'Old name: {oldName}')
print(f'New name: {create_TC_SWIR_NC_CIR_filename(oldName, EVENT_NAME)}')

In [ ]:
# ========================================
# CATEGORIZATION AND MAPPING CONFIG
# ========================================

# Regex patterns to identify file types
CATEGORIZATION_PATTERNS = {
    'trueColor': r'trueColor|truecolor|true_color',
    'colorInfrared': r'colorInfrared|colorIR|color_infrared',
    'naturalColor': r'naturalColor|naturalcolor|natural_color',
    'shortwaveIR': r'shortwaveIR|shortwaveinfrared|shortwaveInfrared'
    # Add patterns for ALL file types you want to process
    # Files not matching any pattern will be marked as 'uncategorized'
}

# Map categories to filename transformation functions
FILENAME_CREATORS = {
    'trueColor': create_TC_SWIR_NC_CIR_filename,
    'colorInfrared': create_TC_SWIR_NC_CIR_filename,
    'naturalColor': create_TC_SWIR_NC_CIR_filename,
    'shortwaveIR': create_TC_SWIR_NC_CIR_filename
    # Must have an entry for each category in CATEGORIZATION_PATTERNS
}

# Output directories for each category
OUTPUT_DIRS = {
    'trueColor': 'Sentinel-2/trueColor',
    'colorInfrared': 'Sentinel-2/colorIR',
    'naturalColor': 'Sentinel-2/naturalColor',
    'shortwaveIR': 'Sentinel-2/shortwaveIR'
    # Must have an entry for each category in CATEGORIZATION_PATTERNS
}

# Nodata values for each category
NODATA_VALUES = {
    'trueColor': 0,
    'colorInfrared': 0,
    'naturalColor': 0,
    'shortwaveIR': 0
    # Must have an entry for each category in CATEGORIZATION_PATTERNS
}

print("Filename transformation functions defined")
print(f"\nCategories configured: {len(CATEGORIZATION_PATTERNS)}")
for category in CATEGORIZATION_PATTERNS.keys():
    print(f"   - {category}")

## Step 4: Preview Transformations

Apply the transformation functions and preview the mapping:

In [ ]:
if not files_df.empty:
    print("Applying filename transformations...\n")

    def categorize_file(filename):
        """Categorize a file based on patterns."""
        for category, pattern in CATEGORIZATION_PATTERNS.items():
            if re.search(pattern, filename, re.IGNORECASE):
                return category
        return 'uncategorized'

    def transform_filename(row):
        """Transform filename based on category."""
        category = row['category']
        original_path = row['original_s3_path']

        if category == 'uncategorized':
            return os.path.basename(original_path)

        if category in FILENAME_CREATORS:
            return FILENAME_CREATORS[category](original_path, EVENT_NAME)

        return os.path.basename(original_path)

    def get_output_path(row):
        """Generate output S3 path."""
        category = row['category']
        new_filename = row['new_filename']

        if category == 'uncategorized':
            return f"{DESTINATION_BASE}/uncategorized/{new_filename}"

        if category in OUTPUT_DIRS:
            return f"{DESTINATION_BASE}/{OUTPUT_DIRS[category]}/{new_filename}"

        return f"{DESTINATION_BASE}/{category}/{new_filename}"

    def get_nodata_value(category):
        """Get nodata value for category."""
        return NODATA_VALUES.get(category, None)

    # Apply transformations
    files_df['category'] = files_df['original_filename'].apply(categorize_file)
    files_df['new_filename'] = files_df.apply(transform_filename, axis=1)
    files_df['output_s3_path'] = files_df.apply(get_output_path, axis=1)
    files_df['nodata_value'] = files_df['category'].apply(get_nodata_value)
    files_df['status'] = 'valid'

    # Check for uncategorized files
    uncategorized = files_df[files_df['category'] == 'uncategorized']
    if not uncategorized.empty:
        files_df.loc[files_df['category'] == 'uncategorized', 'status'] = 'uncategorized'

    # Display preview
    print("TRANSFORMATION PREVIEW")
    print("=" * 80)
    print(f"\nTotal files: {len(files_df)}")
    print(f"Categorized: {len(files_df[files_df['category'] != 'uncategorized'])}")
    print(f"Uncategorized: {len(uncategorized)}")

    # Show category breakdown
    print("\nFiles by category:")
    category_counts = files_df['category'].value_counts()
    for category, count in category_counts.items():
        nodata = NODATA_VALUES.get(category, 'None')
        print(f"   - {category}: {count} files (nodata={nodata})")

    # Show all transformations
    print("\nTransformation details:")
    print("-" * 80)
    for i, row in files_df.iterrows():
        print(f"\n{i+1}. Original: {row['original_filename']}")
        print(f"   Category: {row['category']}")
        print(f"   New name: {row['new_filename']}")
        print(f"   Nodata:   {row['nodata_value']}")
        print(f"   Output:   s3://{BUCKET}/{row['output_s3_path']}")

    if len(uncategorized) > 0:
        print("\nUNCATEGORIZED FILES:")
        print("-" * 80)
        for i, row in uncategorized.iterrows():
            print(f"   - {row['original_filename']}")
        print("\nAdd patterns to CATEGORIZATION_PATTERNS to categorize these files")

    print("\n" + "=" * 80)
else:
    print("No files to process. Check Step 2.")

## Step 5: Export Mapping to CSV

Save the filename mapping to a CSV file:

In [ ]:
if not files_df.empty and SAVE_CSV:
    # Create output directory
    output_path = Path(OUTPUT_DIR) / EVENT_NAME
    output_path.mkdir(parents=True, exist_ok=True)

    csv_filename = f"{EVENT_NAME}-{SUB_PRODUCT_NAME}.csv"
    csv_path = output_path / csv_filename

    # Reorder columns for readability
    column_order = [
        'original_filename',
        'new_filename',
        'category',
        'file_size_gb',
        'nodata_value',
        'status',
        'original_s3_path',
        'output_s3_path'
    ]

    # Save to CSV
    files_df[column_order].to_csv(csv_path, index=False)

    print(f"EXPORT COMPLETE. Saved mapping to: {csv_path}")
    print("=" * 80)
    print(f"   Total records: {len(files_df)}")
    print(f"   Total size:    {files_df['file_size_gb'].sum():.2f} GB")
    print(f"   Valid:         {len(files_df[files_df['status'] == 'valid'])}")
    print(f"   Uncategorized: {len(files_df[files_df['status'] == 'uncategorized'])}")

    # Show nodata value distribution
    nodata_counts = files_df['nodata_value'].value_counts()
    print(f"\n   Nodata values:")
    for nodata, count in nodata_counts.items():
        print(f"      {nodata}: {count} files")

    print("\nYou can now use this CSV in a separate script to perform actual file renaming/copying")

    # Display the DataFrame
    print("\nFull mapping table:")
    print("=" * 80)
    display(files_df[column_order])

elif files_df.empty:
    print("No files to export. Check previous steps.")
else:
    print("CSV export disabled (SAVE_CSV = False)")